# Bulding model pipeline

# CHIRPS Daily Data from https://data.chc.ucsb.edu/products/CHIRPS-2.0/global_daily/netcdf/

In [1]:
#Bulding model pipeline

import os
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
base_dir = r"C:\Users\Tosin\Downloads\BMC"

centroids_path = os.path.join(base_dir, "muni_centroids.csv")  # has all 2469 muni codes
dengue_weekly_path = os.path.join(base_dir, "full_dengue_2020_2026.csv")  # your dengue weekly panel (cases, deaths)
nasa_weekly_path = os.path.join(base_dir, "nasa_power_weekly_muni_2020_2026.csv")  # your weekly NASA climate

out_balanced_path = os.path.join(base_dir, "panel_A_balanced_zero_weeks_2020_2026.csv")

# ------------------------------------------------------------
# 1) Load municipality universe (this defines all spatial units)
# ------------------------------------------------------------
muni = pd.read_csv(centroids_path, dtype={"muni_code": str})
muni["muni_code"] = muni["muni_code"].str.zfill(5)

# ------------------------------------------------------------
# 2) Build a complete ISO week calendar from 2020-01-01 to 2026-12-31
#    Why: we need every week in the study period for every municipality,
#    including weeks with zero cases, to avoid selection bias.
# ------------------------------------------------------------
dates = pd.date_range("2020-01-01", "2026-12-31", freq="D")
cal = pd.DataFrame({"date": dates})

iso = cal["date"].dt.isocalendar()
cal["iso_year"] = iso["year"].astype(int)
cal["iso_week"] = iso["week"].astype(int)

# Keep unique iso year-week combinations in the date range
weeks = cal[["iso_year", "iso_week"]].drop_duplicates().sort_values(["iso_year", "iso_week"]).reset_index(drop=True)

print("Unique ISO weeks in calendar:", weeks.shape[0])
print("ISO year range:", weeks["iso_year"].min(), "to", weeks["iso_year"].max())

# ------------------------------------------------------------
# 3) Cartesian product: all municipalities × all weeks
#    This is the balanced panel we need for valid incidence modeling.
# ------------------------------------------------------------
panel = muni[["muni_code"]].merge(weeks, how="cross")

print("Balanced panel rows (muni × weeks):", panel.shape[0])
print("Balanced panel municipalities:", panel["muni_code"].nunique())

# ------------------------------------------------------------
# 4) Merge dengue weekly counts onto balanced panel
#    Any missing municipality-week in dengue becomes zero cases and zero deaths.
# ------------------------------------------------------------
dengue = pd.read_csv(dengue_weekly_path, dtype={"muni_code": str})
dengue["muni_code"] = dengue["muni_code"].str.zfill(5)

# Keep only the variables we need (change names if yours differ)
needed = ["muni_code", "iso_year", "iso_week", "total_cases", "deaths"]
missing_cols = [c for c in needed if c not in dengue.columns]
if missing_cols:
    raise ValueError(f"Dengue weekly file is missing columns: {missing_cols}")

panel = panel.merge(dengue[needed], on=["muni_code", "iso_year", "iso_week"], how="left")

# Fill missing dengue outcomes with zeros because those are weeks with no reported confirmed cases
panel["total_cases"] = panel["total_cases"].fillna(0).astype(int)
panel["deaths"] = panel["deaths"].fillna(0).astype(int)

# ------------------------------------------------------------
# 5) Merge NASA POWER climate onto balanced panel
#    Missing climate is left as NA for now (rare), later we can drop or impute.
# ------------------------------------------------------------
clim = pd.read_csv(nasa_weekly_path, dtype={"muni_code": str})
clim["muni_code"] = clim["muni_code"].str.zfill(5)

clim_needed = ["muni_code", "iso_year", "iso_week", "t2m_mean", "rh2m_mean"]
missing_cols2 = [c for c in clim_needed if c not in clim.columns]
if missing_cols2:
    raise ValueError(f"NASA weekly file is missing columns: {missing_cols2}")

panel = panel.merge(clim[clim_needed], on=["muni_code", "iso_year", "iso_week"], how="left")

# ------------------------------------------------------------
# 6) Diagnostics: confirm zeros were added and climate coverage is high
# ------------------------------------------------------------
print("\nDiagnostics")
print("Total rows:", panel.shape[0])
print("Weeks with zero cases fraction:", (panel["total_cases"] == 0).mean())
print("Mean cases per muni-week:", panel["total_cases"].mean())

print("Missing temperature fraction:", panel["t2m_mean"].isna().mean())
print("Missing humidity fraction:", panel["rh2m_mean"].isna().mean())

# Check if any municipality is completely missing climate (should be none)
miss_by_muni = panel.groupby("muni_code")["t2m_mean"].apply(lambda s: s.isna().mean())
print("Municipalities with >10% missing temp:", (miss_by_muni > 0.10).sum())

# ------------------------------------------------------------
# 7) Save balanced panel
# ------------------------------------------------------------
panel.to_csv(out_balanced_path, index=False)
print("\nSaved:", out_balanced_path)


Unique ISO weeks in calendar: 366
ISO year range: 2020 to 2026
Balanced panel rows (muni × weeks): 903654
Balanced panel municipalities: 2469

Diagnostics
Total rows: 903654
Weeks with zero cases fraction: 0.9111241692063555
Mean cases per muni-week: 0.752798084222501
Missing temperature fraction: 0.12295081967213115
Missing humidity fraction: 0.12295081967213115
Municipalities with >10% missing temp: 2469

Saved: C:\Users\Tosin\Downloads\BMC\panel_A_balanced_zero_weeks_2020_2026.csv


In [2]:
# Find last available climate week
clim = pd.read_csv(r"C:\Users\Tosin\Downloads\BMC\nasa_power_weekly_muni_2020_2026.csv")

max_year = clim["iso_year"].max()
max_week = clim[clim["iso_year"] == max_year]["iso_week"].max()

print("Last available climate week:", max_year, max_week)


Last available climate week: 2026 8


In [ ]:
## Climate coverage ends at 2026 week 8 which is our study horizon since our dengue data is 2020 upto feb 2026. Everything beyond that would have introduced artificial missing climate.

In [3]:
#import pandas as pd
#import os

base_dir = r"C:\Users\Tosin\Downloads\BMC"

panel_path = os.path.join(base_dir, "panel_A_balanced_zero_weeks_2020_2026.csv")
clim_path = os.path.join(base_dir, "nasa_power_weekly_muni_2020_2026.csv")

panel = pd.read_csv(panel_path, dtype={"muni_code": str})
clim = pd.read_csv(clim_path)

# Determine last valid climate week
max_year = clim["iso_year"].max()
max_week = clim[clim["iso_year"] == max_year]["iso_week"].max()

print("Climate coverage ends at:", max_year, "week", max_week)

# Keep only rows up to that week
panel_restricted = panel[
    (panel["iso_year"] < max_year) |
    ((panel["iso_year"] == max_year) & (panel["iso_week"] <= max_week))
].copy()

print("Rows after restriction:", panel_restricted.shape[0])
print("Missing temperature fraction after restriction:",
      panel_restricted["t2m_mean"].isna().mean())

# Save corrected panel
out_path = os.path.join(base_dir, "panel_A_balanced_zero_weeks_climate_complete.csv")
panel_restricted.to_csv(out_path, index=False)

print("Saved:", out_path)


Climate coverage ends at: 2026 week 8
Rows after restriction: 792549
Missing temperature fraction after restriction: 0.0
Saved: C:\Users\Tosin\Downloads\BMC\panel_A_balanced_zero_weeks_climate_complete.csv


## Model justification

Zero-case fraction earlier was about 91 percent. That tells us:

Dengue risk is sparse and clustered.

This further justifies:

• Negative binomial instead of Poisson
• Municipality random effects
• Possibly zero-inflation testing later

In [4]:
import os
import time
import json
import random
import requests
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
base_dir = r"C:\Users\Tosin\Downloads\BMC"
centroids_path = os.path.join(base_dir, "muni_centroids.csv")
out_dir = os.path.join(base_dir, "chirps_cache")
os.makedirs(out_dir, exist_ok=True)

out_weekly_path = os.path.join(base_dir, "chirps_weekly_muni_2020_2026wk08.csv")

# We align rainfall to the same end week as NASA POWER
end_iso_year = 2026
end_iso_week = 8

# ------------------------------------------------------------
# ClimateSERV CHIRPS API endpoints
# Base URL documented here
# ------------------------------------------------------------
BASE = "https://climateserv.servirglobal.net/chirps"  # base api url :contentReference[oaicite:2]{index=2}
SUBMIT = f"{BASE}/submitDataRequest/"
PROGRESS = f"{BASE}/getDataRequestProgress/"
FETCH = f"{BASE}/getDataFromRequest/"

# ------------------------------------------------------------
# ClimateSERV request settings
# datatype=0 is used in their CHIRPS example request :contentReference[oaicite:3]{index=3}
# intervaltype=0 corresponds to daily in their examples :contentReference[oaicite:4]{index=4}
# operationtype uses codes from getParameterTypes; 4 is sum and 5 is avg in their example output :contentReference[oaicite:5]{index=5}
# For rainfall totals, we want sum over each day for a small polygon around the centroid.
# ------------------------------------------------------------
DATATYPE = 0
INTERVALTYPE_DAILY = 0
OP_SUM = 4

BEGIN = "01/01/2020"
END = "02/22/2026"  # safely covers ISO week 8 of 2026

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def safe_get(url, params, max_tries=8, base_sleep=2.0, timeout=60):
    last = None
    for attempt in range(1, max_tries + 1):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            last = (r.status_code, r.text[:200])
            if r.status_code == 200:
                return r
            if r.status_code in [429, 500, 502, 503, 504]:
                time.sleep(min(base_sleep * (2 ** (attempt - 1)) + random.uniform(0, 1), 120))
                continue
            r.raise_for_status()
        except Exception:
            time.sleep(min(base_sleep * (2 ** (attempt - 1)) + random.uniform(0, 1), 120))
    raise RuntimeError(f"Request failed. Last={last}")

def point_buffer_polygon(lon, lat, delta_deg=0.025):
    """
    ClimateSERV expects EPSG 4326 GeoJSON geometry. :contentReference[oaicite:6]{index=6}
    We approximate a point extraction by using a very small square polygon
    around the centroid.
    delta_deg=0.025 gives about a 5 km half width near the tropics.
    """
    x1, x2 = lon - delta_deg, lon + delta_deg
    y1, y2 = lat - delta_deg, lat + delta_deg
    coords = [[[x1, y1], [x2, y1], [x2, y2], [x1, y2], [x1, y1]]]
    return {"type": "Polygon", "coordinates": coords}

def submit_job(geometry):
    params = {
        "datatype": DATATYPE,
        "begintime": BEGIN,
        "endtime": END,
        "intervaltype": INTERVALTYPE_DAILY,
        "operationtype": OP_SUM,
        "dateType_Category": "default",
        "isZip_CurrentDataType": "false",
        "geometry": json.dumps(geometry),
    }
    r = safe_get(SUBMIT, params, timeout=120)
    job = r.json()
    if isinstance(job, list) and len(job) > 0:
        return job[0]
    raise RuntimeError(f"Unexpected submit response: {job}")

def wait_for_job(job_id, poll_s=4.0, max_minutes=12):
    start_t = time.time()
    while True:
        r = safe_get(PROGRESS, {"id": job_id}, timeout=60)
        prog = float(r.text)
        if prog >= 100.0:
            return
        if prog < 0:
            raise RuntimeError(f"Job error reported for id={job_id}")
        if (time.time() - start_t) > max_minutes * 60:
            raise RuntimeError(f"Job timeout for id={job_id}")
        time.sleep(poll_s)

def fetch_job(job_id):
    r = safe_get(FETCH, {"id": job_id}, timeout=120)
    return r.json()

def to_weekly_from_climateserv(ret_obj):
    """
    ClimateSERV returns a list of granules each with a date string and a value dict. :contentReference[oaicite:7]{index=7}
    We parse dates, then aggregate to ISO week sum.
    """
    data = ret_obj.get("data", [])
    if not data:
        return pd.DataFrame(columns=["iso_year", "iso_week", "rain_mm_week"])

    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])

    # the key inside value matches the operation name, but we requested sum
    # The API example shows value is a dict keyed by the op name :contentReference[oaicite:8]{index=8}
    df["rain"] = df["value"].apply(lambda v: list(v.values())[0] if isinstance(v, dict) and len(v) else np.nan)
    df["rain"] = pd.to_numeric(df["rain"], errors="coerce")
    df = df.dropna(subset=["rain"])

    iso = df["date"].dt.isocalendar()
    df["iso_year"] = iso["year"].astype(int)
    df["iso_week"] = iso["week"].astype(int)

    wk = df.groupby(["iso_year", "iso_week"], as_index=False)["rain"].sum()
    wk = wk.rename(columns={"rain": "rain_mm_week"})
    return wk

# ------------------------------------------------------------
# Load centroids and build request list
# We cache by rounding coordinates to 0.05 degrees to reduce duplicate requests.
# This matches CHIRPS grid resolution order of magnitude. :contentReference[oaicite:9]{index=9}
# ------------------------------------------------------------
cent = pd.read_csv(centroids_path, dtype={"muni_code": str})
cent["muni_code"] = cent["muni_code"].str.zfill(5)

cent["lat_r"] = cent["lat"].round(2)  # about 0.01 deg
cent["lon_r"] = cent["lon"].round(2)
cent["cell_id"] = cent["lat_r"].map(lambda x: f"{x:.2f}") + "_" + cent["lon_r"].map(lambda x: f"{x:.2f}")

cells = cent[["cell_id", "lat_r", "lon_r"]].drop_duplicates().reset_index(drop=True)
print("Municipalities:", cent.shape[0])
print("Unique CHIRPS request cells:", cells.shape[0])

# ------------------------------------------------------------
# Download per cell, cache, convert to weekly
# ------------------------------------------------------------
weekly_cells = []
failed = []

for i, row in cells.iterrows():
    cell_id = row["cell_id"]
    lat = float(row["lat_r"])
    lon = float(row["lon_r"])

    cache_week = os.path.join(out_dir, f"chirps_weekly_{cell_id}.csv")
    if os.path.exists(cache_week):
        wk = pd.read_csv(cache_week, dtype={"iso_year": int, "iso_week": int})
        wk["cell_id"] = cell_id
        weekly_cells.append(wk)
        continue

    try:
        geom = point_buffer_polygon(lon, lat, delta_deg=0.025)
        job_id = submit_job(geom)
        wait_for_job(job_id, poll_s=4.0, max_minutes=12)
        ret = fetch_job(job_id)

        wk = to_weekly_from_climateserv(ret)
        wk["cell_id"] = cell_id

        wk.to_csv(cache_week, index=False)
        weekly_cells.append(wk)

        time.sleep(0.25)

        if (i + 1) % 25 == 0:
            print("Processed cells:", i + 1, "of", cells.shape[0])

    except Exception as e:
        failed.append({"cell_id": cell_id, "lat": lat, "lon": lon, "error": str(e)})
        print("FAILED:", cell_id, str(e)[:120])
        continue

failed_path = os.path.join(base_dir, "chirps_failed_cells.csv")
pd.DataFrame(failed).to_csv(failed_path, index=False)
print("Failed cells saved to:", failed_path)

weekly_cells_df = pd.concat(weekly_cells, ignore_index=True)

# Expand back to municipality
muni_to_cell = cent[["muni_code", "cell_id"]].drop_duplicates()
weekly_muni = muni_to_cell.merge(weekly_cells_df, on="cell_id", how="left")

# Restrict to the same climate horizon as NASA POWER
weekly_muni = weekly_muni[
    (weekly_muni["iso_year"] < end_iso_year) |
    ((weekly_muni["iso_year"] == end_iso_year) & (weekly_muni["iso_week"] <= end_iso_week))
].copy()

weekly_muni = weekly_muni[["muni_code", "iso_year", "iso_week", "rain_mm_week"]]
weekly_muni.to_csv(out_weekly_path, index=False)

print("Saved:", out_weekly_path)
print("Rows:", weekly_muni.shape[0])
print("Missing rain fraction:", weekly_muni["rain_mm_week"].isna().mean())


Municipalities: 2469
Unique CHIRPS request cells: 2469
FAILED: 21.81_-102.30 could not convert string to float: '[0.0]'
FAILED: 22.13_-102.05 could not convert string to float: '[0.0]'
FAILED: 21.90_-102.70 could not convert string to float: '[0.0]'
FAILED: 22.36_-102.30 could not convert string to float: '[0.0]'
FAILED: 21.93_-102.45 could not convert string to float: '[0.0]'
FAILED: 22.10_-102.30 could not convert string to float: '[0.0]'
FAILED: 22.27_-102.34 could not convert string to float: '[0.0]'
FAILED: 22.15_-102.53 could not convert string to float: '[0.0]'
FAILED: 22.24_-102.19 could not convert string to float: '[0.0]'
FAILED: 21.92_-102.04 could not convert string to float: '[0.0]'
FAILED: 22.03_-102.23 could not convert string to float: '[0.0]'
FAILED: 31.10_-115.61 could not convert string to float: '[0.0]'
FAILED: 31.90_-115.28 could not convert string to float: '[0.0]'
FAILED: 32.43_-116.29 could not convert string to float: '[0.0]'
FAILED: 32.41_-116.85 could not con

KeyboardInterrupt: 

In [5]:
!pip -q install earthengine-api geemap

import ee
ee.Authenticate()
ee.Initialize()


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 6.33.5 which is incompatible.
tensorflow-intel 2.18.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.5 which is incompatible.


Enter verification code:  4/1AfrIepBER4ROnDx08lG3VR9tsWwbAd2qZDaaWvVizX9VyueIGlWpVPy8fmU



Successfully saved authorization token.


EEException: Please authorize access to your Earth Engine account by running

earthengine authenticate

in your command line, or ee.Authenticate() in Python, and then retry.